# Solving for the steady state - Thermal explosion

In [ ]:
#    APM41012EP course notebook - Chapter 6 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Solving for the steady state - Thermal explosion
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np

from scipy.sparse import diags
from scipy.sparse.linalg import spsolve, eigs, factorized

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Problem formulation

We want to solve the elliptic problem given by the Poisson equation subject to Dirichlet boundary conditions and to a nonlinear source term of thermal explosion type:

$$
\left\{
\begin{aligned}
-\mathrm{d}_x^2 \theta(x) & = \lambda_{\mathrm{FK}} \exp(\theta(x)) &&x\in \Omega = ]0;2[, \\
        \theta(x) & =  0 && x\in \{0,2\},
\end{aligned}
\right.
$$

where $\lambda_{\mathrm{FK}}$ is the Frank-Kamenetskii parameter. 

In [ ]:
class thermal_explosion_model:
    
    def __init__(self, lamb, xmin, xmax, nx):
        self.lamb = lamb
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx+1)

    def fcn(self, theta):
        lamb = self.lamb
        nx = self.nx
        dx = self.dx
        oneoverdxdx = 1/dx**2
            
        lap = np.zeros(nx)
        lap[0]    = oneoverdxdx * (2*theta[0] - theta[1])
        lap[1:-1] = oneoverdxdx * (-theta[:-2] + 2*theta[1:-1] - theta[2:])
        lap[-1]   = oneoverdxdx * (-theta[-2] + 2*theta[-1])

        return lap - lamb*np.exp(theta)
    
    def jac(self, theta):
        lamb = self.lamb
        nx = self.nx
        dx = self.dx
        diagonals = [np.repeat(2/dx**2, nx) - lamb*np.exp(theta), np.repeat(-1/dx**2, nx-1), np.repeat(-1/dx**2, nx-1)]
        return diags(diagonals, [0, -1, 1])

In [ ]:
def newton(f, jac, x0, tol=1.e-10, max_iter=50, verbose=False):
    
    xk = np.copy(x0)
    
    res = np.zeros(max_iter+1)
    res[0] = np.linalg.norm(f(xk))
    
    incre = np.zeros(max_iter+1)
    incre[0] = 0
    
    # Newton iteration        
    for k in range(max_iter):
        increment = spsolve(jac(xk).tocsr(), -f(xk)) 
        xk = xk + increment
        incre[k+1] = np.linalg.norm(increment)/np.linalg.norm(xk)
        reskp1 = np.linalg.norm(f(xk))
        if (verbose): print(f"Iteration nb {k+1:3d}: ||f(xk)|| = {reskp1:14.8e}")
        res[k+1] = reskp1
        if ( np.linalg.norm(f(xk)) < tol ): break
 
    return xk, res[:k+2], incre[:k+2]

## Newton's method

In [ ]:
xmin = 0.
xmax = 2.
# nb of points including boundary conditions
nxib = 1002
nx = nxib-2
theta_ini = np.zeros(nx)

### Case $\lambda_{\mathrm{FK}} = 0.1$

In [ ]:
# Limit value of lambda = 0.88 
lamb = 0.1
print('***************************************')
print(f"Model for lambda: {lamb}")

tem = thermal_explosion_model(lamb=lamb, xmin=xmin, xmax=xmax, nx=nx)
fcn = tem.fcn
jac = tem.jac

print(f"\nNewton's method:")
max_iter=15
theta_sol, res, increment = newton(fcn, jac, theta_ini, tol=1.e-12, max_iter=max_iter, verbose=True)

jac_eq = jac(theta_sol)

eig_val_min = np.real(eigs(jac_eq, k=1, which='SR')[0])[0]
eig_val_max = np.real(eigs(jac_eq, k=1, which='LR')[0])[0]
print(f"\nCondition number of the Jacobian matrix at equilibrium: {eig_val_max/eig_val_min}")
print(f"\nCondition number of the Laplacian matrix: {(2*(nx+1)/np.pi)**2}")

### Case $\lambda_{\mathrm{FK}} = 0.878$

In [ ]:
# Limit value of lambda = 0.88 
lamb = 0.878
print('\n***************************************')
print(f"Model for lambda: {lamb}")

tem = thermal_explosion_model(lamb=lamb, xmin=xmin, xmax=xmax, nx=nx)
fcn = tem.fcn
jac = tem.jac

print(f"\nNewton's method:")
max_iter=15
theta_sol, res, increment = newton(fcn, jac, theta_ini, tol=1.e-12, max_iter=max_iter, verbose=True)

jac_eq = jac(theta_sol)

eig_val_min = np.real(eigs(jac_eq, k=1, which='SR')[0])[0]
eig_val_max = np.real(eigs(jac_eq, k=1, which='LR')[0])[0]
print(f"\nCondition number of the Jacobian matrix at equilibrium: {eig_val_max/eig_val_min}")
print(f"\nCondition number of the Laplacian matrix: {(2*(nx+1)/np.pi)**2}")

In [ ]:
dx = (xmax-xmin)/(nxib-1)
x = np.linspace(0+dx, 1-dx, nx)

fig = make_subplots(rows=3, cols=1, vertical_spacing=0.12, 
                    subplot_titles=("Solution", "Evolution of the residual", "Evolution of the increment"))
fig.add_trace(go.Scatter(x=x, y=theta_sol, showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=np.arange(res.size), y=res, showlegend=False, mode='lines+markers', line_dash='dot'), row=2, col=1)
fig.add_trace(go.Scatter(x=np.arange(1,increment.size), y=increment[1:], showlegend=False, mode='lines+markers', line_dash='dot'), row=3, col=1)

#create slider
steps = []
for lamb_i in [0.1, 0.5, 0.85, 0.878]:
    tem = thermal_explosion_model(lamb=lamb_i, xmin=xmin, xmax=xmax, nx=nx)
    fcn = tem.fcn
    jac = tem.jac
    theta_sol, res, increment = newton(fcn, jac, theta_ini, tol=1.e-13, max_iter=max_iter, verbose=False)    
    step = dict(method="update", label = f"{lamb_i:.3f}", args=[{"x": [x,np.arange(res.size),np.arange(1,increment.size)], 
                                                                 "y": [theta_sol, res, increment[1:]]}])
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'lambda = '}, active=3, steps=steps)]


fig.update_xaxes(range=[-0.5,increment.size-0.5], row=2)    
fig.update_yaxes(type="log", exponentformat='e', row=2)   
fig.update_xaxes(range=[-0.5,increment.size-0.5], row=3)    
fig.update_yaxes(type="log", exponentformat='e', row=3)    
fig.update_layout(sliders=sliders, height=1200)
fig.show()